# Notebook 3: Movie Chatbot with Web Search + Curated Knowledge

## Learning Objectives
- Use web search for general queries
- Build curated knowledge base from URLs
- Load URLs from use case Excel file
- Create vector store from web content
- Implement two-tool agentic system

## What's New:
- **Tool 1: web_search** - General web search via SERPER API
- **Tool 2: curated_knowledge_search** - Domain-specific knowledge from URLs
- **URL Loading**: Extracts URLs from use case Excel file
- **Vector Store**: FAISS database for curated content
- **Smart Routing**: Agent decides which tool to use

## Prerequisites:
- SERPER_API_KEY in .env
- OPENAI_API_KEY in .env (for embeddings)
- Use case Excel with URLs

## Step 1: Install Required Packages

In [ ]:
%pip install langchain langchain-community langgraph python-dotenv openpyxl pandas langchain-openai google-search-results faiss-cpu --quiet

print("✓ Packages installed")

## Step 2: Import Libraries

In [15]:
import os
import operator
from typing import TypedDict, Annotated, List
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage
from langchain.tools import tool
from langchain_community.utilities import GoogleSerperAPIWrapper
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver

from ai_course_utils import load_use_case_config, display_config

print("✓ All imports successful")

✓ All imports successful


## Step 3: Display Configuration

In [16]:
# Load environment and display config
load_dotenv()
load_dotenv('../.env')  # Also try parent directory

display_config()

API CONFIGURATION (.env file)
LLM Provider:    openai
LLM Model:       gpt-4.1-mini
Temperature:     0.0

API Keys Status:
  OpenAI               ✓ Set
  Google               ✗ Not set
  Mistral              ✗ Not set
  Anthropic            ✗ Not set
  Serper (Web Search)  ✓ Set

Data Files:
  Provide file paths as function parameters
  Example: load_use_case_config('your_file.xlsx')


## Step 4: Load Use Case Configuration

**USER INPUT**: Provide path to your use case Excel file.

In [17]:
# ============================================================================
# USER INPUT: Specify use case file path
# ============================================================================
use_case_file = "../data/use_case_Movie_Recommendation.xlsx"  # ← CHANGE THIS

# Load use case
use_case_config = load_use_case_config(use_case_file)

# Extract system prompt
system_prompt = use_case_config.get("agent_prompt", "You are a helpful movie assistant")

print(f"\n✓ System Prompt loaded")
print(f"Preview: {system_prompt[:150]}...")

✓ Use case loaded: ../data/use_case_Movie_Recommendation.xlsx
  Components: user, agent_prompt, url

✓ System Prompt loaded
Preview: You are a movie expert whose primary goal is to help users with searching for movies. You are friendly and concise. You only provide factual answers t...


## Step 5: Extract and Process URLs from Use Case

This is the key difference from Notebook 2 - we build a curated knowledge base from URLs in the Excel file.

In [18]:
# Extract URLs from use case config
url_string = use_case_config.get("url", "")

if url_string and url_string.strip():
    # Parse URLs (comma-separated)
    urls = [url.strip() for url in url_string.split(",") if url.strip()]
    print(f"\n✓ Found {len(urls)} URLs in use case file:")
    for i, url in enumerate(urls, 1):
        print(f"  {i}. {url}")
else:
    urls = []
    print("\n⚠️ No URLs found in use case file")
    print("Only web_search tool will be available")


✓ Found 3 URLs in use case file:
  1. https://IMDB.com
  2. https://www.themoviedb.org
  3. https://grouplens.org/datasets/movielens/


## Step 6: Build Curated Knowledge Base from URLs

Load content from URLs and create a vector store for semantic search.

In [19]:
url_retriever = None

if urls:
    try:
        print(f"\n📥 Loading content from {len(urls)} URLs...")
        
        # Load documents from URLs
        loader = WebBaseLoader(urls)
        documents = loader.load()
        print(f"  ✓ Loaded {len(documents)} documents")
        
        # Split documents into chunks
        text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=1000,
            chunk_overlap=200
        )
        splits = text_splitter.split_documents(documents)
        print(f"  ✓ Created {len(splits)} text chunks")
        
        # Create vector store
        embeddings = OpenAIEmbeddings()
        vectorstore = FAISS.from_documents(splits, embeddings)
        print(f"  ✓ Vector store created")
        
        # Create retriever
        url_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
        print(f"  ✓ Retriever ready (returns top 3 matches)")
        
        print("\n✅ Curated knowledge base built successfully!")
        
    except Exception as e:
        print(f"\n❌ Error building knowledge base: {str(e)}")
        print("Only web_search tool will be available")
        url_retriever = None
else:
    print("\n⚠️ Skipping knowledge base creation (no URLs provided)")


📥 Loading content from 3 URLs...
  ✓ Loaded 3 documents
  ✓ Created 15 text chunks
  ✓ Vector store created
  ✓ Retriever ready (returns top 3 matches)

✅ Curated knowledge base built successfully!


## Step 7: Define Tools

Define both tools that the agent can use.

In [20]:
# Tool 1: General Web Search (Always available)
@tool
def web_search(query: str):
    """
    Use this for general web searches for real-time information like weather,
    news, current events, or topics not covered by the curated knowledge base.
    
    Best for:
    - Current information (today's news, weather, etc.)
    - General queries about any topic
    - Recent events or releases
    """
    search = GoogleSerperAPIWrapper()
    return search.run(query)

# Tool 2: Curated Knowledge Search (Conditionally available)
@tool
def curated_knowledge_search(query: str):
    """
    Use this to find specific, expert information from a trusted knowledge base
    built from curated URLs (like IMDB, TMDB, etc.).
    
    This is the BEST tool for:
    - Detailed, domain-specific questions
    - Information from trusted movie databases
    - Expert content from the configured URLs
    
    DO NOT use this for general or current events queries.
    """
    if url_retriever is None:
        return "Error: The curated knowledge base is not available for this use case."
    
    docs = url_retriever.invoke(query)
    results = []
    for doc in docs:
        source = doc.metadata.get('source', 'N/A')
        content = doc.page_content
        results.append(f"Source: {source}\nContent: {content}")
    
    return "\n\n---\n\n".join(results)

# Build tools list based on availability
if url_retriever is not None:
    tools = [web_search, curated_knowledge_search]
    print("✓ Tools configured: web_search, curated_knowledge_search")
else:
    tools = [web_search]
    print("✓ Tools configured: web_search only")

✓ Tools configured: web_search, curated_knowledge_search


## Step 8: Initialize LLM with Tools

In [21]:
# Load LLM from environment
from ai_course_utils import load_llm_from_env

llm = load_llm_from_env()

# Bind tools to LLM
model = llm.bind_tools(tools)

print("✓ LLM initialized with tools")

🤖 Loading LLM: openai / gpt-4.1-mini
✓ LLM initialized with tools


## Step 9: Define Agent State and Graph

In [22]:
# Define agent state
class AgentState(TypedDict):
    messages: Annotated[List[BaseMessage], operator.add]

# Agent node: calls the model
def call_model(state: AgentState):
    messages = [SystemMessage(content=system_prompt)] + state["messages"]
    response = model.invoke(messages)
    return {"messages": [response]}

# Decision function: continue to tools or end
def should_continue(state: AgentState):
    last_message = state["messages"][-1]
    if last_message.tool_calls:
        return "tools"
    return END

print("✓ Agent state and nodes defined")

✓ Agent state and nodes defined


## Step 10: Build and Compile Agent Graph

In [23]:
# Create tool node
tool_node = ToolNode(tools)

# Build workflow
workflow = StateGraph(AgentState)

# Add nodes
workflow.add_node("agent", call_model)
workflow.add_node("tools", tool_node)

# Set entry point
workflow.set_entry_point("agent")

# Add conditional edges
workflow.add_conditional_edges(
    "agent",
    should_continue,
    {
        "tools": "tools",
        END: END
    }
)

# Add edge from tools back to agent
workflow.add_edge("tools", "agent")

# Compile with memory
memory = MemorySaver()
app = workflow.compile(checkpointer=memory)

print("✓ Agent workflow compiled")

✓ Agent workflow compiled


## Step 11: Create Query Function

In [24]:
def run_query(query: str, thread_id: str = "default", verbose: bool = True):
    """
    Run a query through the agent.
    
    Args:
        query: User's question
        thread_id: Conversation thread ID
        verbose: Show detailed execution
    """
    if verbose:
        print(f"\n{'='*70}")
        print(f"User Query: {query}")
        print(f"{'='*70}")
    
    config = {"configurable": {"thread_id": thread_id}}
    
    # Track which tools are used
    tools_used = []
    
    for event in app.stream(
        {"messages": [HumanMessage(content=query)]},
        config,
        stream_mode="values"
    ):
        last_message = event["messages"][-1]
        
        # Track tool usage
        if hasattr(last_message, 'tool_calls') and last_message.tool_calls:
            for tool_call in last_message.tool_calls:
                tool_name = tool_call['name']
                if tool_name not in tools_used:
                    tools_used.append(tool_name)
                    if verbose:
                        print(f"  🔧 Using tool: {tool_name}")
    
    if verbose:
        if tools_used:
            print(f"  ✓ Tools used: {', '.join(tools_used)}")
        else:
            print(f"  ℹ️ No tools used (answered directly)")
        print()
    
    # Get final response
    final_message = event["messages"][-1]
    return final_message.content

print("✓ Query function ready")
print("\n🎬 Agent with Web Search + Curated Knowledge ready!")

✓ Query function ready

🎬 Agent with Web Search + Curated Knowledge ready!


## Step 12: Test the Agent

Test with different types of queries to see tool selection.

In [26]:
# Test 1: Current information (should use web_search)
print("\nTest 1: Current Information")
query1 = "What are the current top box office movies as listed in IMDB?"
response1 = run_query(query1, thread_id="test1")
print(f"Response: {response1}")


Test 1: Current Information

User Query: What are the current top box office movies as listed in IMDB?
  🔧 Using tool: curated_knowledge_search
  ✓ Tools used: curated_knowledge_search

Response: It seems I couldn't retrieve the current top box office movies specifically from IMDb at this moment. However, I have the latest top box office list from other reliable sources if you'd like. Would you prefer that, or do you want me to try again later? Also, do you have any particular genre or type of movie you're interested in?


In [12]:
# Test 2: Domain-specific (should use curated_knowledge_search if available)
print("\n" + "="*70)
print("\nTest 2: Domain-Specific Information")
query2 = "Tell me about Christopher Nolan's filmography"
response2 = run_query(query2, thread_id="test2")
print(f"Response: {response2}")



Test 2: Domain-Specific Information

User Query: Tell me about Christopher Nolan's filmography
  🔧 Using tool: curated_knowledge_search
  ✓ Tools used: curated_knowledge_search

Response: It seems I couldn't retrieve a detailed list of Christopher Nolan's filmography from the current source. However, I can share some of his most notable films from memory:

- Memento (2000)
- Insomnia (2002)
- Batman Begins (2005)
- The Prestige (2006)
- The Dark Knight (2008)
- Inception (2010)
- The Dark Knight Rises (2012)
- Interstellar (2014)
- Dunkirk (2017)
- Tenet (2020)
- Oppenheimer (2023)

Would you like more detailed information about any specific film or about his career in general? Or are you interested in recommendations based on his style?


In [13]:
# Test 3: General knowledge (might not use tools)
print("\n" + "="*70)
print("\nTest 3: General Knowledge")
query3 = "What genre is The Godfather?"
response3 = run_query(query3, thread_id="test3")
print(f"Response: {response3}")



Test 3: General Knowledge

User Query: What genre is The Godfather?
  🔧 Using tool: curated_knowledge_search
  ✓ Tools used: curated_knowledge_search

Response: The Godfather is primarily classified as a Crime and Drama genre film. Would you like to know more about its plot or cast? Or are you interested in similar movies?


## Step 13: Interactive Testing

Try your own queries!

In [14]:
# YOUR TURN: Test with your own query
my_query = "What are the Oscar nominations for Best Picture in 2026?"

print(f"\nUser: {my_query}")
response = run_query(my_query, thread_id="my_session")
print(f"\nBot: {response}")


User: What are the Oscar nominations for Best Picture in 2026?

User Query: What are the Oscar nominations for Best Picture in 2026?
  🔧 Using tool: web_search
  ✓ Tools used: web_search


Bot: The Oscar nominations for Best Picture in 2026 include the following films:
- Bugonia
- F1
- Frankenstein
- Hamnet
- Marty Supreme
- One Battle After Another
- The Secret Agent
- Sentimental Value

Would you like more details about any of these films?


## Summary

### What We Built:
✅ Two-tool agentic system  
✅ Web search for general/current queries  
✅ Curated knowledge search from URLs  
✅ Automatic URL extraction from Excel  
✅ Vector store for semantic search  
✅ Smart tool routing by LLM  

### Key Differences from Notebook 2:
- **Notebook 2**: Only web_search
- **Notebook 3**: web_search + curated_knowledge_search
- **Data Source**: URLs from use case Excel file
- **Vector Store**: FAISS for semantic search

### Tools Available:
1. **web_search**: Current information, general queries
2. **curated_knowledge_search**: Domain expert knowledge from URLs

### When Each Tool is Used:
- **web_search**: Current events, news, general topics
- **curated_knowledge_search**: Detailed info from trusted sources (IMDB, TMDB, etc.)

### Next Steps:
**Notebook 4**: Data analysis with Pandas and visualization  
**Notebook 5**: RAG with PDF documents (Oscars 2026)  
**Notebook 6**: TAG with genre taxonomy  
**Notebook 7**: Complete system (RAG + TAG)

### Experiments to Try:
1. Add different URLs to your use case file
2. Compare responses with/without curated knowledge
3. Test which queries trigger which tools
4. Modify the system prompt to guide tool selection